In [1]:
import sys
sys.path.append('./../')
! pip install lifelines
! pip install torchdiffeq

### Importing the functions

In [2]:
from src.utils import *
from src.data import *
from src.model import *
from src.trainer import *
from src.visualisation import *

/home/jorge/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/jorge/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


### Loading configuration along with creating the numpy arrays for the data

In [3]:
configuration_file = './../experiments/sample_config.json'
configuration = load_json_config(configuration_file)

# Process imputed data
#process_imputed_data(configuration)

### Defining the hyperparameters for the experiments 

In [4]:
# features define whether maggic or maggic_plus
configurations = [
    {
        'features': 'maggic', 'feature_size': 13, 'mlp_hidden_sizes': [128, 256, 128], 'mlp_output_size': 64, 
        'ode_hidden_size': 64, 'ode_num_layers': 2, 'ode_batch_norm': False,'time_nums': 50
    },
    {
        'features': 'maggic_plus', 'feature_size': 19, 'mlp_hidden_sizes': [128, 256, 128], 'mlp_output_size': 64, 
        'ode_hidden_size': 64, 'ode_num_layers': 2, 'ode_batch_norm': False,'time_nums': 50
    }
]

In [9]:
import torch
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

def run_experiment(config, data_folder, model_path, device):
    train_filepath = os.path.join(data_folder, f"train_{config['features']}.npz")
    test_filepath = os.path.join(data_folder, f"valid_{config['features']}.npz")

    model = MLP_SODEN(config)
    model = model.to(device)
    model.suffix = config['features']
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2, verbose=True)
    
    num_epochs = 10
    patience = 3

    results = main_training_loop(model, train_filepath, test_filepath, model_path, optimizer, num_epochs, patience, scheduler, device)
    return results

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

results = {}

for hyper_params in configurations:
    results[hyper_params['features']] = run_experiment(hyper_params, configuration['data_folder'], configuration['model_folder'], device)
    tempprc, tempauroc, val_loss, _ = results[hyper_params['features']]
    print(f"Final results for {hyper_params['features']}:")
    print(f"TempPRC: {tempprc:.4f}, TempAUROC: {tempauroc:.4f}, Validation Loss: {val_loss:.4f}")

    model = load_model(hyper_params, configuration['model_folder']).to(device)
    e, ph, t = get_validation_results(model, configuration['data_folder'], hyper_params['features'])
    save_outputs(hyper_params['features'], t, e, ph, [1, 12, 24, 36, 48])



FileNotFoundError: [Errno 2] No such file or directory: './../data/train_maggic.npz'

In [7]:
data, MAPlabel, color = create_config()

# Assuming process_results is defined elsewhere
outc, outprc = process_results(data, MAPlabel)
outc = [[x[0], x[1], x[1]] for x in outc][::-1]
names = [MAPlabel[x] for x in list(data)][::-1]

plot_forest(outc, names)

FileNotFoundError: [Errno 2] No such file or directory: './../outputs/maggic_plus_12.npz'

In [8]:
constrain_map = create_constrain_map(data)
plot_calibration(data, constrain_map, color, MAPlabel)

FileNotFoundError: [Errno 2] No such file or directory: './../outputs/maggic_plus_12.npz'

In [42]:
from data import convert_datatypes
from data import extract_and_split
path="/home/jorge/work_dir/federado/flcore_AI4HF/data/sample.parquet"
path="/home/jorge/work_dir/federado/flcore_AI4HF/data/ampliado.parquet"

field_mapping = {
"patid": "patid", "pracid": "pracid", "gender": "gender", "label": "label", "time2event": "time2event", 
"age": "age", "bmi": "bmi", "smoke": "smoke", "systolic": "systolic", "creatinine": "creatinine", 
"bb": "bb", "acei_arb": "acei_arb", "diab": "diab", "copd": "copd", "EF": "EF", "sodium": "sodium", 
"nyha": "nyha", "hftime": "hftime", "af": "af", "stroke": "stroke", "mi": "mi"}
    
int_columns = ["gender", "age", "label", "time2event", "smoke", "bb", "acei_arb", "diab", "copd", "nyha", "hftime", "af", "stroke", "mi"]
float_columns = ["bmi", "systolic", "creatinine", "sodium", "EF"]
maggic_columns = ["gender", "age", "smoke", "hftime", "systolic", "creatinine", "bmi", "nyha", "EF", "copd", "diab", "bb", "acei_arb"]
label_columns = ["time2event", "label"]

df=convert_datatypes(path,field_mapping,int_columns,float_columns)
extract_and_split(df, maggic_columns, label_columns,  "suffix", "./")

###################################

from data import get_dataloader
train_dataloader, _ = get_dataloader("/home/jorge/work_dir/federado/flcore_AI4HF/src/Models/MLP/train_suffix.npz", batch_size=512, std=1)




LABEL ['time2event', 'label']
FEAT ['gender', 'age', 'smoke', 'hftime', 'systolic', 'creatinine', 'bmi', 'nyha', 'EF', 'copd', 'diab', 'bb', 'acei_arb']
label, featues
convertido numpy
guardando ... 


In [43]:
data=[]
batches = []
labels = []
temp = {}
for _,i in enumerate(train_dataloader):
    print("__")
    batch, label = i
    batches.append(batch)
    labels.append(label)
    temp["batch"] = batch
    temp["label"] = label
    data.append(temp)
    #print(batch,label)

__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__
__


In [44]:
# dict_keys(['t', 'init_cond', 'features', 'index'])
batch.keys()

dict_keys(['t', 'init_cond', 'features', 'index'])

In [47]:
len(data)

18

In [46]:
type(data[0]["label"])

torch.Tensor

In [37]:
import pandas as pd
path="/home/jorge/work_dir/federado/flcore_AI4HF/data/sample.parquet"
pa=pd.read_parquet(path)

In [38]:
pa

,gender,age,smoke,hftime,systolic,creatinine,bmi,nyha,EF,copd,...,bb,acei_arb,sodium,af,mi,stroke,pci,cabg,time2event,label
0,1,30,1,1,249.159070,1509.353878,46.424248,3,0.505565,1,...,0,1,122.343198,0,1,1,0,0,2,0
1,0,20,1,1,163.965100,894.488209,47.088680,3,0.298876,0,...,1,0,155.005492,1,1,1,0,0,54,1
2,0,58,0,0,166.599984,111.335908,33.236032,2,0.294999,1,...,0,1,132.533567,0,0,1,1,1,21,1
3,0,42,0,1,88.111235,1418.441137,41.845333,1,0.536705,0,...,1,0,115.904617,0,0,1,1,1,18,1
4,0,45,0,0,268.719870,1821.803941,43.706214,0,0.352473,1,...,0,1,128.679070,1,0,0,0,0,24,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1,33,1,1,86.195883,1405.292843,40.731499,4,0.278726,0,...,1,1,129.354099,1,1,1,1,0,37,0
996,1,61,1,0,140.625484,1900.430316,32.445738,1,0.364990,1,...,0,0,121.637784,0,1,0,1,0,25,1
997,0,54,0,0,150.400494,1264.531830,29.058720,3,0.305752,0,...,1,0,156.266008,1,1,1,1,1,41,1
998,1,74,1,1,254.418991,194.827662,31.231485,4,0.511806,0,...,0,0,119.914619,1,0,0,1,0,37,1


In [40]:
df_concatenado = pd.concat([df]*10, ignore_index=True)
df_concatenado.to_parquet("ampliado.parquet")